## Intstall XGBoost

In [0]:
%pip install xgboost
%restart_python

## Convert Delta Table -> Pandas DataFrame

In [0]:
import pandas as pd
df_from_spark = spark.table("multimodal_adni_living_model_ready_SHAP_no_MOCA")
pdf = df_from_spark.toPandas()

In [0]:
pdf.shape, pdf.head()
# check shape

## Fix Class Imbalance  |  90.5%/9.5% -> 50%/50%

Changing from a large class imbalance of 9.5% positive to 50% positive

4105-432 to 432-432

In [0]:
import pandas as pd
# Separate minority and majority classes
df_1 = pdf[pdf['y_mmse_binary'] == 1]
df_0 = pdf[pdf['y_mmse_binary'] == 0]
 
# Randomly sample from majority class
df_0_sampled = df_0.sample(n=len(df_1), random_state=42)
 
# Combine and shuffle
balanced_df = (
    pd.concat([df_1, df_0_sampled])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
 
print(balanced_df['y_mmse_binary'].value_counts())
pdf = balanced_df

Quick Check

In [0]:
print(pdf.head())

## Split into X and y for Model

In [0]:
X = pdf.drop(columns=["y_mmse_binary"])
y = pdf["y_mmse_binary"].astype(int)

### Quick checks for Data Leakage
Check for IDs, to make sure (first check still included NPID, NPIDSEV, and GDAFRAID)
- Check these variables first before removing, make sure they are actual identifiers
- NPID: Just happens to have ID, but its question 4(D) of the NPI
- NPIDSEV: The severity of the above question
- GDAFRAID: Also not an ID, but the 4th (D) question

### One-hot encoding to handle strings for XGBoost

In [0]:
import pandas as pd
X_enc = pd.get_dummies(X,drop_first=False, dummy_na=True)
X_enc.shape

VIF Check

In [0]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = X_enc[["MMLTR5_W", "MMHAND"]].dropna()

# Convert boolean to numeric for VIF calculation
X = X.astype(float)

vif_data = pd.DataFrame()
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print(vif_data)

In [0]:
# Check for IDs
[c for c in X.columns if "id" in c.lower()]
# Check for Scores
[c for c in X.columns if "score" in c.lower()]
# WORLDSCORE, MMSCORE, NPISCORE removed

In [0]:
X = X_enc
y = y.astype(int) # making sure its still integers only.
y.value_counts()

In [0]:
print(X)

## Train/Test split

In [0]:
from sklearn.model_selection import train_test_split
# Train/test split, stratified to keep class ratio consistent

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    stratify=y,
    random_state=42, # 42 because its the answer to the universe
    test_size=0.3
)

In [0]:
# Check for class ratios
print(y_train.value_counts())
print()
print(y_test.value_counts())
print()

## Random Forest Test


In [0]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

# Base model
rf = RandomForestClassifier(
    random_state=42,
    n_jobs=-1
)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"]
}

# Grid search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="RF_GridSearch"):

    grid_search.fit(X_train, y_train)

    rf_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = rf_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = rf_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = rf_best_model.predict(X_train)
    y_test_pred_class = rf_best_model.predict(X_test)

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other Metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)
    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)
    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)
    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)
    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.sklearn.log_model(rf_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap:", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

## Train Model (XGBoost)
Looks like its Overfitting based on the gap between train and test auc (0.05)

In [0]:
import mlflow
import mlflow.xgboost
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score

# Base model
xgb = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=1
)

# Hyperparameter grid
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

# Grid search
grid_search = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="XGB_GridSearch"):

    grid_search.fit(X_train, y_train)

    xgb_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = xgb_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = xgb_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = xgb_best_model.predict(X_train)
    y_test_pred_class = xgb_best_model.predict(X_test)

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)

    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)

    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)

    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)

    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.xgboost.log_model(xgb_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap:", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

## SVM Test

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import roc_auc_score, average_precision_score
from mlflow.models.signature import infer_signature

# Pipeline: impute -> scale -> SVM
svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("svm", SVC(probability=True, random_state=42))
])

# Reasonable grid size
param_grid = {
    "svm__kernel": ["linear", "rbf"],
    "svm__C": [0.1, 1, 10],
    "svm__gamma": ["scale", 0.01, 0.001]
}

grid_search = GridSearchCV(
    estimator=svm_pipeline,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=5,
    n_jobs=-1,
    verbose=1
)

with mlflow.start_run(run_name="SVM_GridSearch"):

    grid_search.fit(X_train, y_train)

    svm_best_model = grid_search.best_estimator_

    # Probability predictions
    y_train_pred_proba = svm_best_model.predict_proba(X_train)[:, 1]
    y_test_pred_proba = svm_best_model.predict_proba(X_test)[:, 1]

    # Class predictions
    y_train_pred_class = svm_best_model.predict(X_train)
    y_test_pred_class = svm_best_model.predict(X_test)

    # AUC
    train_auc = roc_auc_score(y_train, y_train_pred_proba)
    test_auc = roc_auc_score(y_test, y_test_pred_proba)
    auc_gap = train_auc - test_auc

    # Other Metrics
    train_pr = average_precision_score(y_train, y_train_pred_proba)
    test_pr = average_precision_score(y_test, y_test_pred_proba)
    train_accuracy = accuracy_score(y_train, y_train_pred_class)
    test_accuracy = accuracy_score(y_test, y_test_pred_class)
    train_precision = precision_score(y_train, y_train_pred_class)
    test_precision = precision_score(y_test, y_test_pred_class)
    train_recall = recall_score(y_train, y_train_pred_class)
    test_recall = recall_score(y_test, y_test_pred_class)
    train_f1 = f1_score(y_train, y_train_pred_class)
    test_f1 = f1_score(y_test, y_test_pred_class)

    # Log best params
    mlflow.log_params(grid_search.best_params_)

    # Log metrics
    mlflow.log_metric("train_auc", train_auc)
    mlflow.log_metric("test_auc", test_auc)
    mlflow.log_metric("auc_gap", auc_gap)
    mlflow.log_metric("train_pr_auc", train_pr)
    mlflow.log_metric("test_pr_auc", test_pr)
    mlflow.log_metric("best_cv_auc", grid_search.best_score_)

    mlflow.log_metric("train_accuracy", train_accuracy)
    mlflow.log_metric("test_accuracy", test_accuracy)
    mlflow.log_metric("train_precision", train_precision)
    mlflow.log_metric("test_precision", test_precision)
    mlflow.log_metric("train_recall", train_recall)
    mlflow.log_metric("test_recall", test_recall)
    mlflow.log_metric("train_f1", train_f1)
    mlflow.log_metric("test_f1", test_f1)

    mlflow.sklearn.log_model(svm_best_model, "best_model")

print("Best Params:", grid_search.best_params_)
print("Best CV AUC:", grid_search.best_score_)

print("Train AUC:", train_auc)
print("Test AUC:", test_auc)
print("AUC Gap", auc_gap)
print("Train PR AUC:", train_pr)
print("Test PR AUC:", test_pr)

print("Train Accuracy:", train_accuracy)
print("Test Accuracy:", test_accuracy)
print("Train Precision:", train_precision)
print("Test Precision:", test_precision)
print("Train Recall:", train_recall)
print("Test Recall:", test_recall)
print("Train F1:", train_f1)
print("Test F1:", test_f1)

### Evaluate All

In [0]:
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, roc_auc_score

def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Probabilities (needed for ROC-AUC)
    if hasattr(model, "predict_proba"):
        y_test_prob = model.predict_proba(X_test)[:, 1]
    else:
        # fallback (rare case)
        y_test_prob = model.decision_function(X_test)

    # Metrics
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_test_pred),
        "Precision": precision_score(y_test, y_test_pred),
        "Recall": recall_score(y_test, y_test_pred),
        "F1 Score": f1_score(y_test, y_test_pred),
        "ROC-AUC": roc_auc_score(y_test, y_test_prob)
    }

    # Confusion matrix
    cm = confusion_matrix(y_test, y_test_pred)

    print(f"\n{name} Results:")
    for k, v in metrics.items():
        if k != "Model":
            print(f"{k}: {v:.4f}")

    print("\nConfusion Matrix:")
    print(cm)

    return metrics, cm

In [0]:
rf_metrics, rf_cm = evaluate_model("Random Forest", rf_best_model, X_train, y_train, X_test, y_test)

xgb_metrics, xgb_cm = evaluate_model("XGBoost", xgb_best_model, X_train, y_train, X_test, y_test)

svm_metrics, svm_cm = evaluate_model("SVM", svm_best_model, X_train, y_train, X_test, y_test)

In [0]:
results_df = pd.DataFrame([rf_metrics, xgb_metrics, svm_metrics])
print(results_df)

In [0]:
%pip install shap
%restart_python

In [0]:
import mlflow
import mlflow.xgboost
import pandas as pd
import json
import shap
import numpy as np

# Replace Run ID when you want to check a different run, should be at the bottom of the mlflow logging step above
# Before: f0436711bfa24095b86d8df85110c670
# After 2/27 variable change: 36efe52695df49608ece51cebb470c16
RUN_ID = "36efe52695df49608ece51cebb470c16"

# Load model
model = mlflow.xgboost.load_model(f"runs:/{RUN_ID}/model")

# Download artifacts from MLFlow run. Feature_cols, X_test, and y_test
feature_cols_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="feature_columns.json"
)
X_test_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="X_test.parquet"
)
y_test_path = mlflow.artifacts.download_artifacts(
    run_id=RUN_ID, artifact_path="y_test.parquet"
)

with open(feature_cols_path, "r") as f:
    feature_cols = json.load(f)

X_test = pd.read_parquet(X_test_path)[feature_cols]
y_test = pd.read_parquet(y_test_path)["y"].astype(int)

# ---- TreeSHAP ----
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Handle version differences: sometimes list per class (suggested by ai)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

shap.summary_plot(shap_values, X_test)

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Use whichever matrix you want to analyze (X_train, X_test, or full X)
df = X_test.copy()

# Keep only numeric cols (important if any categorical slipped in)
df = df.select_dtypes(include=[np.number])


# Correlation type:
# - "pearson" for linear relationships
# - "spearman" for monotonic relationships (prob better for clinical scores)
corr = df.corr(method="spearman")

# Plot
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr.values, aspect="auto")

ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.index)))
ax.set_xticklabels(corr.columns, rotation=90)
ax.set_yticklabels(corr.index)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Spearman correlation")

ax.set_title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## Next Steps:
- Feature reduction, VIF, feature selection further
- Look at feature selection, SHAP, VIF, Correlation Map


VIF Check